<a href="https://colab.research.google.com/github/ascii5833/Deep-Learning-Mastery/blob/main/Codes/Transformers/Transformers_Encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import torch
import torch.nn as nn
import math
import torch.optim as optim

In [44]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self, embed_dim, n_heads, ff_dim, dropout = 0.2):
    super(TransformerEncoderLayer, self).__init__()

    #multiheaded self-attention layer
    self.atn = nn.MultiheadAttention(
        embed_dim = embed_dim,
        num_heads = n_heads,
        dropout = dropout,
        batch_first = True

    )

    #forward network
    self.ffn  = nn.Sequential(
        nn.Linear(embed_dim, ff_dim),
        nn.ReLU(),
        nn.Linear(ff_dim, embed_dim)
    )

    #layer norm
    self.norm1 = nn.LayerNorm(embed_dim)
    self.norm2 = nn.LayerNorm(embed_dim)

    #dropout layers
    self.drop1 = nn.Dropout(dropout)
    self.drop2 = nn.Dropout(dropout)


  #forward pass
  def forward(self, X):
    #inp: (batch, seqlength, embeddings)

    #self-attention
    att, _ = self.atn(X, X, X)

    #skip connection
    X = self.norm1(X + self.drop1(att))

    #feed forward network
    ff = self.ffn(X)

    #skip connection again
    X = self.norm2(X + self.drop2(ff))

    return X







In [45]:
batch_size = 2
seq_length = 10
embed_dim = 64

x = torch.randn(batch_size, seq_length, embed_dim)

encoder = TransformerEncoderLayer(embed_dim = 64, n_heads = 4, ff_dim = 256)

output = encoder(x)

print(output.shape)



torch.Size([2, 10, 64])


<h1>Mini-BERT pre-training example </h1>

In [46]:
#function to mask tokens
MASK_TOKEN = 104
def mask_tokens(input_ids, mask_prob = 0.3, mask_token = 104):
  #clone the input tensor of ids
  labels = input_ids.clone()

  #create a random of same size as input
  rand = torch.rand(labels.shape)
  # if random less than mask_prob make it true
  mask = rand < mask_prob
  #now where it is true, we will make it the mask
  input_ids = input_ids.clone()
  input_ids[mask] = mask_token
  #loss only where theres mask
  labels[~mask] = -100

  return input_ids, labels


tokens = torch.tensor([101, 120, 130, 140], requires_grad = False)

inp_ids, labels = mask_tokens(tokens)
print(f'Tokens are {tokens}')
print(f'Input ids are: {inp_ids} \n Labels are: {labels}')

Tokens are tensor([101, 120, 130, 140])
Input ids are: tensor([104, 120, 130, 140]) 
 Labels are: tensor([ 101, -100, -100, -100])


<h1> Mini-BERT </h1>

In [84]:
class MiniBERT(nn.Module):
  def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers):
    super(MiniBERT, self).__init__()

    #create embedding layer
    self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = embed_dim)

    #layers -> stack them using modulelist
    self.layers = nn.ModuleList([
        TransformerEncoderLayer(embed_dim= embed_dim, n_heads=num_heads, ff_dim=ff_dim, dropout=0.5)
        for _ in range(num_layers)
    ])

    #output head (we softmax over the outputs) i.e softmax over the vocabulary to predict which word more likely to be predicted
    self.encoder_head = nn.Linear(embed_dim, vocab_size)

  #forward pass
  def forward(self, X):
    X = self.embedding(X)

    #forward pass through layers
    for layer in self.layers:
      X = layer(X)

    #get the logits
    logits = self.encoder_head(X)

    return logits



In [97]:
model = MiniBERT(vocab_size = 30000, embed_dim = 256, num_heads = 8, ff_dim = 1024, num_layers = 20)

print(model)


MiniBERT(
  (embedding): Embedding(30000, 256)
  (layers): ModuleList(
    (0-19): 20 x TransformerEncoderLayer(
      (atn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (ffn): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (drop1): Dropout(p=0.5, inplace=False)
      (drop2): Dropout(p=0.5, inplace=False)
    )
  )
  (encoder_head): Linear(in_features=256, out_features=30000, bias=True)
)


In [109]:

#optimizer
optimizer = optim.Adam(model.parameters(), lr = 5e-4)
#loss function (-100 to ignore unmasked positions)
criterion = nn.CrossEntropyLoss(ignore_index = -100)




<h1>Dataset</h1>

In [110]:
from torch.utils.data import Dataset, DataLoader

In [111]:
class TextDataset(Dataset):
  def __init__(self, texts, tokenizer, max_len = 16):
    self.texts = texts
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, index):
    text = self.texts[index]

    encoding = self.tokenizer(
        text,
        padding = 'max_length',
        truncation = True,
        max_length = self.max_len,
        return_tensors = 'pt'
    )

    input_ids = encoding['input_ids'].squeeze(0)

    return input_ids


In [112]:
!pip install transformers

In [113]:
from transformers import BertTokenizer

In [114]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [115]:
texts = [
    "I love transformers",
    "Deep learning is fun",
    "Dogs are amazing",
    "Aahan is the best",
    "You are the worst",
    "Vash the stampeded is the trigun hero",
    "Attack on titan ending sucks",
    "Ben 10 is nostalgia",
    "Messi is the goat",
    "Ronaldo is the camel"
]

dataset = TextDataset(texts, tokenizer)

In [116]:
loader = DataLoader(dataset, batch_size = 4, shuffle = True)

In [117]:
MASK_TOKEN_ID = tokenizer.mask_token_id

In [118]:
special_tokens = [
    tokenizer.cls_token_id,
    tokenizer.sep_token_id,
    tokenizer.pad_token_id
]
print(special_tokens)

def mask_tokens(input_ids, mask_prob = 0.3, mask_token = 104, skip_tokens = []):
  #clone the input tensor of ids
  labels = input_ids.clone()

  #create a random of same size as input
  rand = torch.rand(labels.shape)
  # if random less than mask_prob make it true
  mask = rand < mask_prob
  #dont mask special tokens
  for token in skip_tokens:
    mask = mask & (input_ids != token)
  #now where it is true, we will make it the mask
  input_ids = input_ids.clone()
  input_ids[mask] = mask_token
  #loss only where theres mask
  labels[~mask] = -100

  return input_ids, labels


tokens = torch.tensor([101, 120, 130, 140], requires_grad = False)

inp_ids, labels = mask_tokens(tokens, mask_prob=0.4, mask_token = 104, skip_tokens = special_tokens)
print(f'Tokens are {tokens}')
print(f'Input ids are: {inp_ids} \n Labels are: {labels}')

[101, 102, 0]
Tokens are tensor([101, 120, 130, 140])
Input ids are: tensor([101, 104, 130, 140]) 
 Labels are: tensor([-100,  120, -100, -100])


In [120]:
from torch.optim.lr_scheduler import CosineAnnealingLR

<h1>Training Loop </h1>

In [124]:
n_epochs = 30
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=n_epochs
)
for epoch in range(n_epochs):
  model.train()
  total_loss = 0
  for batch in loader:
    masked_inp, labels = mask_tokens(batch, mask_prob=0.15, mask_token = MASK_TOKEN_ID, skip_tokens = special_tokens)
    #check whether the batch is broken i.e no mask present (no 104s present)
    if (labels != -100).sum() == 0:
      print('batch broken, skipping')
      continue
    #forward pass
    logits = model(masked_inp)
    loss = criterion(
        logits.view(-1, logits.size(-1)),
        labels.view(-1)
    )
    if torch.isnan(logits).any():
      print("NaN in logits")
      break

    if torch.isinf(logits).any():
      print("Inf in logits")
      break

    if torch.isnan(loss):
      print("NaN in loss")
      break
    #backward pass
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 2. clip gradients

    optimizer.step()

    total_loss += loss.item()

  print(f'Epoch {epoch+1}, Loss: {total_loss/len(loader)}')
  scheduler.step()


Epoch 1, Loss: 3.5295752684275308
batch broken, skipping
Epoch 2, Loss: 2.284823179244995
Epoch 3, Loss: 3.816332737604777
Epoch 4, Loss: 4.303509553273519
batch broken, skipping
Epoch 5, Loss: 2.2846028804779053
Epoch 6, Loss: 3.576080878575643
Epoch 7, Loss: 3.026785929997762
Epoch 8, Loss: 3.9432570139567056
Epoch 9, Loss: 4.074323256810506
Epoch 10, Loss: 3.2520035107930503
Epoch 11, Loss: 3.6780900955200195
batch broken, skipping
batch broken, skipping
Epoch 12, Loss: 1.0249779224395752
Epoch 13, Loss: 3.674970547358195
Epoch 14, Loss: 4.19630233446757
Epoch 15, Loss: 3.24443785349528
Epoch 16, Loss: 3.449715773264567
Epoch 17, Loss: 3.1340903441111245
Epoch 18, Loss: 3.9464728832244873
Epoch 19, Loss: 4.265390316645305
batch broken, skipping
Epoch 20, Loss: 2.364859422047933
Epoch 21, Loss: 3.3059750398000083
batch broken, skipping
Epoch 22, Loss: 1.6745814482371013
Epoch 23, Loss: 3.313518444697062
batch broken, skipping
Epoch 24, Loss: 2.7296335697174072
Epoch 25, Loss: 2.74372